# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The purpose of this baseline is to identify content that should be prioritized for a content refresh.

This rule assumes that content is more valuable to review when it:
- has not been updated for at least 180 days,
- still receives meaningful search visibility (500 or more impressions in the last 30 days),
- and is not ranking near the top of search results (average position greater than 10).

The rule is intentionally simple and transparent so that every recommendation can be explained to a human. This baseline will later be compared with a machine learning model.

### Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| STALE_VISIBLE | Old content that still receives meaningful search visibility |
| STALE | Old content but limited recent visibility |
| LOW_POSITION | Content ranks outside the top positions |
| REVIEW | General review candidate |

### Action Label

Refresh Content

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import pandas as pd


df = pd.read_csv("content_refresh_anonymized.csv")

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_last_30d"] >= 500).astype(int)
low_position = (df["avg_position"] > 10).astype(int)

df["baseline_score"] = (
    stale * 2 +
    visible +
    low_position
)

def reason(row):

    if row["days_since_last_update"] >= 180 and row["impressions_last_30d"] >= 500:
        return "STALE_VISIBLE"

    if row["days_since_last_update"] >= 180:
        return "STALE"

    if row["avg_position"] > 10:
        return "LOW_POSITION"

    return "REVIEW"

df["reason_code"] = df.apply(reason, axis=1)

df["action_label"] = "Refresh Content"

df = df.sort_values(
    "baseline_score",
    ascending=False
)

output = df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
]

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

output.head(20)

,content_id,baseline_score,reason_code,action_label
16751,content_cf56e2e2e282,4,STALE_VISIBLE,Refresh Content
5327,content_fe16a55cd13d,4,STALE_VISIBLE,Refresh Content
12045,content_c2d929d83eaa,4,STALE_VISIBLE,Refresh Content
698,content_b16bd7307b39,4,STALE_VISIBLE,Refresh Content
7021,content_1bfaa38ff26c,4,STALE_VISIBLE,Refresh Content
16514,content_7368877ea310,4,STALE_VISIBLE,Refresh Content
26810,content_ecb6215e79fd,4,STALE_VISIBLE,Refresh Content
21268,content_0a91db491d14,4,STALE_VISIBLE,Refresh Content
665,content_e444c00065bd,3,STALE,Refresh Content
7045,content_026a1e2a82fd,3,STALE,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

| Rank | Action | Reason Code | Confidence | What would make it wrong |
|------|---------|-------------|------------|--------------------------|
|1|Refresh Content|STALE_VISIBLE|High|The page may be evergreen and still meeting user intent despite its age.|
|2|Refresh Content|STALE_VISIBLE|High|Recent visibility may already be improving without requiring a refresh.|
|3|Refresh Content|STALE_VISIBLE|High|The topic may have naturally declining demand rather than outdated content.|
|4|Refresh Content|STALE_VISIBLE|High|Performance could be affected by external search changes rather than content quality.|
|5|Refresh Content|STALE_VISIBLE|High|The content may intentionally target a niche audience with stable performance.|
|6|Refresh Content|STALE_VISIBLE|High|A recent update may not yet be reflected in the available data.|
|7|Refresh Content|STALE_VISIBLE|High|Ranking could improve without content changes because of seasonal demand.|
|8|Refresh Content|STALE_VISIBLE|High|The page could already satisfy user intent despite ranking outside the top results.|
|9|Refresh Content|STALE|Medium|Low visibility may reduce the impact of refreshing this page.|
|10|Refresh Content|STALE|Medium|Traffic may remain low because of limited search demand.|
|11|Refresh Content|STALE|Medium|Content may already be accurate even though it is old.|
|12|Refresh Content|STALE|Medium|A low search volume topic may not justify the effort.|
|13|Refresh Content|STALE|Medium|The page may have historical importance but limited business value.|
|14|Refresh Content|STALE|Medium|The low ranking could be caused by strong competitors rather than outdated content.|
|15|Refresh Content|STALE|Medium|Refreshing may not improve performance if user intent has changed.|
|16|Refresh Content|STALE|Medium|The page may no longer be strategically important.|
|17|Refresh Content|STALE|Medium|Search visibility may be too low to benefit from an update.|
|18|Refresh Content|STALE|Medium|The page may already be scheduled for future maintenance.|
|19|Refresh Content|STALE|Medium|The ranking issue may be technical rather than content-related.|
|20|Refresh Content|STALE|Medium|The page may perform adequately for its intended audience despite its age.|

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The baseline intentionally uses simple rules, so some recommendations may not be ideal.

Examples of weak picks include:

- Old pages that continue performing well because they contain evergreen content.
- Pages with low search demand where refreshing content is unlikely to produce measurable improvements.
- Pages with low rankings caused by strong competition instead of outdated content.

These cases highlight where a machine learning model may outperform the rule-based baseline.

## Leakage Check

The baseline uses only observable features available at prediction time:

- days_since_last_update
- impressions_last_30d
- avg_position

The following fields were intentionally **not** used:

- trend_direction
- trend_pct
- client identifiers
- content identifiers
- any future-window information

No product flags or future-derived information were used when computing the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.